# DSCI 100 Project Planning Stage (Individual)

In [1]:
### Run this cell before continuing.

import altair as alt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import set_config
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import GridSearchCV, cross_validate
from sklearn.neighbors import KNeighborsClassifier

# Simplify working with large datasets in Altair
alt.data_transformers.enable('vegafusion')

DataTransformerRegistry.enable('vegafusion')

## (1) Data Description Section


#### Players Dataset
---

In [2]:
players = pd.read_csv("players.csv")
players.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 196 entries, 0 to 195
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   experience        196 non-null    object 
 1   subscribe         196 non-null    bool   
 2   hashedEmail       196 non-null    object 
 3   played_hours      196 non-null    float64
 4   name              196 non-null    object 
 5   gender            196 non-null    object 
 6   age               196 non-null    int64  
 7   individualId      0 non-null      float64
 8   organizationName  0 non-null      float64
dtypes: bool(1), float64(3), int64(1), object(4)
memory usage: 12.6+ KB


In [3]:
players.head()

,experience,subscribe,hashedEmail,played_hours,name,gender,age,individualId,organizationName
0,Pro,True,f6daba428a5e19a3d47574858c13550499be23603422e6...,30.3,Morgan,Male,9,NaN,NaN
1,Veteran,True,f3c813577c458ba0dfef80996f8f32c93b6e8af1fa9397...,3.8,Christian,Male,17,NaN,NaN
2,Veteran,False,b674dd7ee0d24096d1c019615ce4d12b20fcbff12d79d3...,0.0,Blake,Male,17,NaN,NaN
3,Amateur,True,23fe711e0e3b77f1da7aa221ab1192afe21648d47d2b4f...,0.7,Flora,Female,21,NaN,NaN
4,Regular,True,7dc01f10bf20671ecfccdac23812b1b415acd42c2147cb...,0.1,Kylie,Male,21,NaN,NaN


##### Description

- There are 196 observations
- 9 variables
| Variable Name       | Type            | Description                                               | Potential Issues        |
| ------------------  | -----------     | ----------------------------------------                  | ----------------------- |
| `experience`        | categorical     | How much experience each player has with the game         | None                    |
| `subscribe`         | categorical     | If a player has subscribed to the game-related newsletter | None                    |
| `hashedEmail`       | categorical     | Hashed version of each players' unique email              | None                    |
| `played_hours`      | numerical       | How long each player has played the game in hours         | None                    |
| `name`              | categorical     | The first name of each player                             | First and last name would be better in case there are duplicate first names |
| `gender`            | categorical     | Gender assigned at birth of each player                   | None                    |
| `age`               | numerical       | The age of each of player                                 | None                    |
| `individualId`      | numerical       | Unique identifier for each player                         | The whole column is empty |
| `organizationName`  | categorical     | What organization each player belongs to                  | The whole column is empty |

> The data must've been collected during the sign up process 

#### Sessions Dataset
---

In [4]:
sessions = pd.read_csv("sessions.csv")
sessions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1535 entries, 0 to 1534
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   hashedEmail          1535 non-null   object 
 1   start_time           1535 non-null   object 
 2   end_time             1533 non-null   object 
 3   original_start_time  1535 non-null   float64
 4   original_end_time    1533 non-null   float64
dtypes: float64(2), object(3)
memory usage: 60.1+ KB


In [5]:
sessions.head()

,hashedEmail,start_time,end_time,original_start_time,original_end_time
0,bfce39c89d6549f2bb94d8064d3ce69dc3d7e72b38f431...,30/06/2024 18:12,30/06/2024 18:24,1.719770e+12,1.719770e+12
1,36d9cbb4c6bc0c1a6911436d2da0d09ec625e43e6552f5...,17/06/2024 23:33,17/06/2024 23:46,1.718670e+12,1.718670e+12
2,f8f5477f5a2e53616ae37421b1c660b971192bd8ff77e3...,25/07/2024 17:34,25/07/2024 17:57,1.721930e+12,1.721930e+12
3,bfce39c89d6549f2bb94d8064d3ce69dc3d7e72b38f431...,25/07/2024 03:22,25/07/2024 03:58,1.721880e+12,1.721880e+12
4,36d9cbb4c6bc0c1a6911436d2da0d09ec625e43e6552f5...,25/05/2024 16:01,25/05/2024 16:12,1.716650e+12,1.716650e+12


##### Description

- There are 1535 observations
- 5 variables
| Variable Name        | Type            | Description                                               | Potential Issues        |
| ------------------   | -----------     | ----------------------------------------                  | ----------------------- |
| `hashedEmail`        | categorical     | Hashed version of each players' unique email              | None                    |
| `start_time`         | numerical       | Human readable time stamp                                 | None                    |
| `end_time`           | numerical       | Human readable time stamp                                 | None                    |
| `original_start_time`| numerical       | Unix epoch time in milliseconds                           | None                    |
| `orignal_end_time`   | numerical       | Unix epoch time in milliseconds                           | None                    |

> This data must've been collected before and after each play session which was probably recorded in a database 

## (2) Question Section 

> Question 2: We would like to know which "kinds" of players are most likely to contribute a large amount of data so that we can target those players in our recruiting efforts.

- The `players.csv` dataset provides personal and experience related information about each player

- The `sessions.csv` dataset provides information about each play session (including start and end times)

- By merging these datasets through shared hashedEmail, I can calculate how much `total_playtime` each player contributed

- Then create create a new categorical variable that represents if a player is a `high_data_player` based on their `total_playtime`, making it my response variable of interest, and using player characteristics, like `experience`, `age`, `gender`, and `subscribe`, as my explanatory variables.


## (3) Explanatory Data Analysis and Visualization

> No minimum necessary wrangling needed as the dataset is already tidy

#### Players Dataset
---

In [6]:
players.head()

,experience,subscribe,hashedEmail,played_hours,name,gender,age,individualId,organizationName
0,Pro,True,f6daba428a5e19a3d47574858c13550499be23603422e6...,30.3,Morgan,Male,9,NaN,NaN
1,Veteran,True,f3c813577c458ba0dfef80996f8f32c93b6e8af1fa9397...,3.8,Christian,Male,17,NaN,NaN
2,Veteran,False,b674dd7ee0d24096d1c019615ce4d12b20fcbff12d79d3...,0.0,Blake,Male,17,NaN,NaN
3,Amateur,True,23fe711e0e3b77f1da7aa221ab1192afe21648d47d2b4f...,0.7,Flora,Female,21,NaN,NaN
4,Regular,True,7dc01f10bf20671ecfccdac23812b1b415acd42c2147cb...,0.1,Kylie,Male,21,NaN,NaN


In [7]:
players_plot1 = alt.Chart(players).mark_bar().encode(
    x = alt.X("gender").title("Gender"),
    y = alt.Y("played_hours").title("Played time in hours"),
    color = alt.Color("experience").title("Experience with the game")
).properties(title='Distribution of Players')
players_plot1

alt.Chart(...)

- The player group mainly consists of females, males, and non-binary
- The male player group have the highest combined of played time
- Those on the higher end of played time tend to label themselves as amateurs, whereas those on the lower end as regulars
- For some reason, those with low played time label themselves as veterans

#### Sessions Dataset
---

In [8]:
sessions.head()

,hashedEmail,start_time,end_time,original_start_time,original_end_time
0,bfce39c89d6549f2bb94d8064d3ce69dc3d7e72b38f431...,30/06/2024 18:12,30/06/2024 18:24,1.719770e+12,1.719770e+12
1,36d9cbb4c6bc0c1a6911436d2da0d09ec625e43e6552f5...,17/06/2024 23:33,17/06/2024 23:46,1.718670e+12,1.718670e+12
2,f8f5477f5a2e53616ae37421b1c660b971192bd8ff77e3...,25/07/2024 17:34,25/07/2024 17:57,1.721930e+12,1.721930e+12
3,bfce39c89d6549f2bb94d8064d3ce69dc3d7e72b38f431...,25/07/2024 03:22,25/07/2024 03:58,1.721880e+12,1.721880e+12
4,36d9cbb4c6bc0c1a6911436d2da0d09ec625e43e6552f5...,25/05/2024 16:01,25/05/2024 16:12,1.716650e+12,1.716650e+12


In [9]:
sessions['start_time'] = pd.to_datetime(sessions['start_time'], format='%d/%m/%Y %H:%M')
sessions['end_time'] = pd.to_datetime(sessions['end_time'], format='%d/%m/%Y %H:%M')
sessions['duration_minutes'] = (sessions['end_time'] - sessions['start_time']).dt.total_seconds() / 60

sessions_plot = alt.Chart(sessions).mark_bar().encode(
    x=alt.X('duration_minutes')
        .bin(maxbins=20)
        .title('Session Duration (minutes)'),
    y=alt.Y('count()').title('Number of Sessions')
).properties(title='Distribution of Session Durations')

sessions_plot

alt.Chart(...)

- Most sessions last for 20 minutes
- The longer the duration of the session the smaller amount of sessions played

## (4) Methods and Plan

- For the question: "We would like to know which "kinds" of players are most likely to contribute a large amount of data so that we can target those players in our recruiting efforts." I will use a trained k-nearest neighbours (KNN) classifier model
- This method is appropriate because my response variable, `high_data_player`, is categorical. KNN can capture complex, non-linear relationships between features such as `experience`, `age`, `gender`, and `subscribe` without requiring strong distributional assumptions
- Some potential limitations include KNN’s sensitivity to irrelevant or highly correlated features and to the choice of k. It can also perform poorly if the data is very imbalanced or contains many categorical variables
- To compare and select models I will evaluate the classification accuracy across different values of k. Cross-validation will be used to determine the optimal k and to prevent overfitting
- Before applying the model, I will merge the datasets by `hashedEmail`, compute each player’s total playtime, and create the categorical variable `high_data_player`. Then, I will split the data into training (70%) and testing (30%) sets before fitting/training the model and tuning it through cross validation, using `GridSearchCV`